In [1]:
import numpy as np
import json
import torch
from torchsummary import summary
from tqdm import tqdm
from pathlib import Path

In [2]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version used by PyTorch: {torch.version.cuda}")
else:
    print("CUDA is not available. PyTorch is using CPU.")

CUDA is not available. PyTorch is using CPU.


In [3]:
from src.models.base_model import base_model
from src.utils.metrics import AverageMeter, CombinedLoss, DiceLoss, dice_coefficient, iou_score, pixel_accuracy

Пути к каталогам.

In [4]:
config_dir = Path("./src/config/")
model_name = 'base-model'

config_path = config_dir / "config.json"
assert config_path.exists(), f"Config not found: {config_path}"
with open(config_path, "r") as f:
    general_config = json.load(f)

model_config_path = config_dir / f"{model_name}-config.json"
assert model_config_path.exists(), f"Config not found: {model_config_path}"
with open(model_config_path, "r") as f:
    model_config = json.load(f)

dataset_config_path = config_dir / f"{model_config['dataset_name']}-config.json"
assert dataset_config_path.exists(), f"Config not found: {dataset_config_path}"
with open(dataset_config_path, "r") as f:
    dataset_config = json.load(f)
    
data_path = Path(general_config['data_dir']) / dataset_config['dataset_name']

In [5]:
device = torch.device(general_config["device"].lower() if torch.cuda.is_available() else 'cpu')

# Тест архитектуры

In [6]:
mdl_input_size = model_config['input_size']

model = base_model(
    in_channels = mdl_input_size[0],
    out_channels = 4,
    features = model_config['feature_list'],
    #device = self.device
    )
model = model.to(device)
model.eval()

test_input = torch.randn(1, *mdl_input_size).to(device)
test_output = model(test_input)

model_size = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model size: {model_size}")
print(f"  Input:  {test_input.shape}")
print(f"  Output: {test_output.shape}")
summary(model, tuple(mdl_input_size))

Encoder features by level: [512, 1024]
Model size: 1058308
  Input:  torch.Size([1, 2, 512])
  Output: torch.Size([1, 4, 512])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1             [-1, 512, 512]           3,584
       BatchNorm1d-2             [-1, 512, 512]           1,024
              ReLU-3             [-1, 512, 512]               0
         MaxPool1d-4             [-1, 512, 256]               0
   ConvTranspose1d-5            [-1, 1024, 512]       1,049,600
            Conv1d-6               [-1, 4, 512]           4,100
Total params: 1,058,308
Trainable params: 1,058,308
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 11.02
Params size (MB): 4.04
Estimated Total Size (MB): 15.06
----------------------------------------------------------------


## dataloader

In [7]:
SEED = np.random.randint(0, 10000)
torch.manual_seed(SEED)
np.random.seed(SEED)

In [29]:
from src.dataloaders.ZerosPolesDataset import TransformsConfig, ZerosPolesDataset
from torch.utils.data import DataLoader
# Setting seeds.
def worker_init_fn(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)
    
train_transforms = TransformsConfig(
    #crop_ratio=[1.0, 1.0],
    time_delay=[0.0, 1e-9],
    noise_level=[5e-3, 30e-3],
    noise_reduce=2,
    gain=[0.9, 1.1]
)
val_transforms = None

train_loader = DataLoader(
    ZerosPolesDataset(
        dataset_dir = Path(general_config['data_dir']) / dataset_config['dataset_name'],
        split = 'train',
        transforms=train_transforms
        ),
    batch_size=model_config["batch_size"],
    shuffle=True,
    num_workers=0,#model_config["workers"],
    worker_init_fn=worker_init_fn,
    pin_memory=True)

val_loader = DataLoader(
    ZerosPolesDataset(
        dataset_dir = Path(general_config['data_dir']) / dataset_config['dataset_name'],
        split = 'val',
        transforms=val_transforms
        ),
    batch_size=model_config["batch_size"],
    shuffle=False,
    num_workers=model_config["workers"],
    pin_memory=True)

In [34]:
# 1. Initialize accumulators (use float64 for numerical stability)
channel_sum = None
channel_sum_sq = None
total_samples = 0  # Total number of scalar values per channel

# 2. Iterate over dataset
with torch.no_grad():
    for data_tuple in tqdm(val_loader, desc="Calculating Stats"):
        inputs = data_tuple[0].to(device)
        
        inputs = inputs.float()
        B, C, L = inputs.shape  # [Batch, Channels, Length]
        
        # Initialize on first batch (handles dynamic channel count)
        if channel_sum is None:
            channel_sum = torch.zeros(C, dtype=torch.float64, device=device)
            channel_sum_sq = torch.zeros(C, dtype=torch.float64, device=device)
        
        # 3. Accumulate sum and sum of squares per channel
        # Sum over Batch (0) and Length (2), keep Channel (1)
        channel_sum += inputs.sum(dim=(0, 2)).double()
        channel_sum_sq += (inputs ** 2).sum(dim=(0, 2)).double()
        
        # Count total samples per channel: Batch * Length
        total_samples += B * L

# 4. Compute Final Mean and Std
mean = channel_sum / total_samples
# Variance = E[X²] - (E[X])²
var = (channel_sum_sq / total_samples) - (mean ** 2)

mean = mean.cpu()
std = torch.sqrt(var).cpu()

print(f"Mean: {mean.numpy()}")
print(f"Std:  {std.numpy()}")

Calculating Stats: 100%|██████████| 20/20 [00:07<00:00,  2.58it/s]

Mean: [529.12203564 496.96538726]
Std:  [11079.7596183  11102.14110049]


## Run 1:
- Mean: [ 984.91466768 2397.26128974]
- Std:  [ 55818.49198814 218841.79374928]

## Run 2:
- Mean: [1001.5774273 2397.346644 ]
- Std:  [ 57678.93105483 218281.70283202]

## Run 3:
- Mean: [1014.77855625 2394.29139941]
- Std:  [ 59264.07932032 217862.15627679]

In [10]:
with torch.no_grad():
    
    inputs, masks, _ = next(iter(val_loader))
    inputs, masks = inputs.to(device), masks.to(device)
    
    outputs = model(inputs)
        
    print(dice_coefficient(outputs.detach(), masks.detach()))
    print(iou_score(outputs.detach(), masks.detach()))
    
    '''
    for data_tuple in tqdm(val_loader, desc="Val"):
        
        inputs, masks = data_tuple[0].to(device), data_tuple[1].to(device)
        
        outputs = model(inputs)
        
        print(dice_coefficient(outputs.detach(), masks.detach()))
        print(iou_score(outputs.detach(), masks.detach()))
    '''

d:\git\ITMO\frequency-response-encoder\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


0.0008542835712432861
0.000428000814281404
